# Document Retrieval, Deep Dive, Abstention

Ce notebook répond à trois questions, dans cet ordre:
1. retrouve-t-on le bon document BOFIP ?
2. extrait-on un bon passage dans ce document ?
3. sait-on déjà quand il faut s'abstenir ?

La priorité reste volontairement:
- `#1` bon document
- `#2` bon extrait
- `#3` abstention


In [1]:
from pathlib import Path
import json
import sys

PROJECT_ROOT = Path.cwd().resolve().parent
SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from bofip_cleanroom.jsonio import read_jsonl

def load_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))

queries_v2 = read_jsonl(PROJECT_ROOT / "data" / "interim" / "retrieval_queries_sample_1000_v2.jsonl")
doc_eval_v2 = load_json(PROJECT_ROOT / "data" / "reports" / "phase3_doc_lexical_eval_raw_docs_sample_1000__retrieval_queries_sample_1000_v2.json")
two_stage_body_v2 = load_json(PROJECT_ROOT / "data" / "reports" / "phase3_two_stage_probe_sample_1000_v2_body.json")
deep_dive_v2 = load_json(PROJECT_ROOT / "data" / "reports" / "phase3_deep_dive_probe_retrieval_queries_sample_1000_v2_body.json")
abstention_v2 = load_json(PROJECT_ROOT / "data" / "reports" / "phase3_abstention_audit_retrieval_queries_sample_1000_v2.json")

query_map = {row["id"]: row for row in queries_v2}
doc_eval_map = {row["id"]: row for row in doc_eval_v2["results"]}
two_stage_map = {row["id"]: row for row in two_stage_body_v2["rows"]}
deep_dive_map = {row["id"]: row for row in deep_dive_v2["rows"]}
abstain_map = {row["id"]: row for row in abstention_v2["rows"]}


## 1. Vue d'ensemble


In [2]:
{
    "query_count_v2": len(queries_v2),
    "behavior_counts": abstention_v2["behavior_counts"],
    "doc_retrieval_metrics_supported_only": doc_eval_v2["metrics"],
    "abstention_best_rule": abstention_v2["best_rule"],
}


{'query_count_v2': 75,
 'behavior_counts': {'answer': 57, 'abstain': 18},
 'doc_retrieval_metrics_supported_only': {'hit@1': 0.9333,
  'hit@3': 0.9778,
  'hit@5': 0.9778},
 'abstention_best_rule': {'rule_family': 'combined_uncovered_ratio',
  'uncovered_ratio_min': 0.5,
  'accuracy': 0.8267,
  'abstain_precision': 0.7778,
  'abstain_recall': 0.3889,
  'answer_recall': 0.9649,
  'tp': 7,
  'tn': 55,
  'fp': 2,
  'fn': 11}}

## 2. Objectif numéro 1: le bon document BOFIP

Ici on regarde les rares vrais ratés du retrieval documentaire.


In [3]:
[
    {
        "id": row["id"],
        "pattern": row["pattern"],
        "query": row["query"],
        "expected_boi": row["expected_boi"],
        "returned_boi": row["returned_boi"][:5],
    }
    for row in doc_eval_v2["results"]
    if row.get("supported_query") and not row.get("hit@1")
]


[{'id': 'q01',
  'pattern': 'near_neighbor',
  'query': 'une JEI peut-elle demander le remboursement immédiat du CIR',
  'expected_boi': 'BOI-BIC-CHAMP-80-20-20-20-20240703',
  'returned_boi': ['BOI-RES-SJ-000014-20210309',
   'BOI-TVA-SECT-80-60-10-50-20120912',
   'BOI-ENR-DG-50-20-20160203',
   'BOI-TVA-DED-50-20-10-20150506',
   'BOI-TVA-DED-50-20-20-20210224']},
 {'id': 'q29',
  'pattern': 'explicit_reference',
  'query': 'États et territoires non coopératifs droit conventionnel',
  'expected_boi': 'BOI-INT-DG-20-50-20210224',
  'returned_boi': ['BOI-INT-DG-20-50-10-20210224',
   'BOI-INT-DG-20-50-20210224',
   'BOI-INT-DG-20-20-40-20120912',
   'BOI-INT-DG-20-30-20250115',
   'BOI-INT-CVB-USA-10-20-20-20150812']},
 {'id': 'q30',
  'pattern': 'topic_specific',
  'query': 'prélèvement forfaitaire obligatoire applicable aux produits de placement à revenu fixe',
  'expected_boi': 'BOI-RPPM-RCM-30-20-40-20220630',
  'returned_boi': ['BOI-RPPM-RCM-30-20-70-20191220',
   'BOI-RPPM-RCM-3

## 3. Helpers d'affichage


In [4]:
def doc_trace(query_id):
    row = doc_eval_map[query_id]
    return {
        "query_id": query_id,
        "query": row["query"],
        "pattern": row["pattern"],
        "expected_boi": row.get("expected_boi"),
        "returned_boi": row["returned_boi"][:5],
        "top_hits": row["top_hits"][:5],
    }

def two_stage_trace(query_id):
    row = two_stage_map[query_id]
    return {
        "query_id": query_id,
        "query": row["query"],
        "expected_boi": row.get("expected_boi"),
        "expected_behavior": query_map[query_id].get("expected_behavior"),
        "stage1_docs": row["document_hits"][:3],
        "stage2_chunks": row["chunk_hits"][:6],
    }

def deep_dive_trace(query_id):
    row = deep_dive_map[query_id]
    return {
        "query_id": query_id,
        "query": row["query"],
        "expected_boi": row.get("expected_boi"),
        "expected_behavior": row.get("expected_behavior"),
        "stage1_docs": row["document_hits"][:3],
        "stage2_top_chunks": row["top_chunk_hits"][:4],
        "stage3_expanded_context": row["expanded_context"][:10],
    }

def abstention_trace(query_id):
    row = abstain_map[query_id]
    return {
        "query_id": query_id,
        "query": row["query"],
        "pattern": row["pattern"],
        "expected_behavior": row["expected_behavior"],
        "predicted_behavior": row["predicted_behavior"],
        "decision_correct": row["decision_correct"],
        "top_doc_boi": row["top_doc_boi"],
        "doc_margin": row["doc_margin"],
        "title_overlap_ratio": row["title_overlap_ratio"],
        "chunk_overlap_ratio": row["chunk_overlap_ratio"],
        "combined_uncovered_ratio": row["combined_uncovered_ratio"],
        "top_chunk_text": row["top_chunk_text"],
    }


## 4. Cas réels pour l'objectif `#1`

Succès nets, voisinages thématiques, et ratés documentaires.


In [5]:
doc_trace("q02")


{'query_id': 'q02',
 'query': "quelles dépenses de fonctionnement sont éligibles au crédit d'impôt recherche",
 'pattern': 'near_neighbor',
 'expected_boi': 'BOI-BIC-RICI-10-10-20-25-20250813',
 'returned_boi': ['BOI-BIC-RICI-10-10-20-25-20250813',
  'BOI-BIC-RICI-10-10-20-50-20250813',
  'BOI-BIC-RICI-10-10-20-40-20250813',
  'BOI-IS-RICI-10-10-20-20210721',
  'BOI-BIC-RICI-10-10-10-30-20160706'],
 'top_hits': [{'rank': 1,
   'score': 39.793,
   'boi_reference': 'BOI-BIC-RICI-10-10-20-25-20250813',
   'chunk_id': '10594-PGP__document_title',
   'chunk_kind': 'document_title',
   'section_path': 'BIC - Réductions et crédits d’impôt - Crédits d’impôt - Crédit d’impôt recherche - Dépenses de recherche éligibles - Autres dépenses de fonctionnement'},
  {'rank': 2,
   'score': 31.7873,
   'boi_reference': 'BOI-BIC-RICI-10-10-20-50-20250813',
   'chunk_id': '6507-PGP__document_title',
   'chunk_kind': 'document_title',
   'section_path': 'BIC - Réductions et crédits d’impôt - Crédits d’impô

In [6]:
doc_trace("q12")


{'query_id': 'q12',
 'query': 'détermination du redevable de la TVA pour les livraisons de biens et prestations de services',
 'pattern': 'topic_specific',
 'expected_boi': 'BOI-TVA-DECLA-10-10-20-20251022',
 'returned_boi': ['BOI-TVA-DECLA-10-10-20-20251022',
  'BOI-TVA-DECLA-10-10-20200323',
  'BOI-TVA-DECLA-10-10-30-20-20200902',
  'BOI-TVA-CHAMP-30-30-30-20190109',
  'BOI-TVA-CHAMP-20-50-40-20-20211222'],
 'top_hits': [{'rank': 1,
   'score': 52.6082,
   'boi_reference': 'BOI-TVA-DECLA-10-10-20-20251022',
   'chunk_id': '3218-PGP__document_title',
   'chunk_kind': 'document_title',
   'section_path': 'TVA - Régimes d’imposition et obligations déclaratives et comptables - Redevable de la taxe - Livraisons de biens et prestations de services - Détermination du redevable'},
  {'rank': 2,
   'score': 46.868,
   'boi_reference': 'BOI-TVA-DECLA-10-10-20200323',
   'chunk_id': '3163-PGP__document_title',
   'chunk_kind': 'document_title',
   'section_path': "TVA - Régimes d'imposition et 

In [7]:
doc_trace("q29")


{'query_id': 'q29',
 'query': 'États et territoires non coopératifs droit conventionnel',
 'pattern': 'explicit_reference',
 'expected_boi': 'BOI-INT-DG-20-50-20210224',
 'returned_boi': ['BOI-INT-DG-20-50-10-20210224',
  'BOI-INT-DG-20-50-20210224',
  'BOI-INT-DG-20-20-40-20120912',
  'BOI-INT-DG-20-30-20250115',
  'BOI-INT-CVB-USA-10-20-20-20150812'],
 'top_hits': [{'rank': 1,
   'score': 47.7398,
   'boi_reference': 'BOI-INT-DG-20-50-10-20210224',
   'chunk_id': '12855-PGP__document_title',
   'chunk_kind': 'document_title',
   'section_path': 'INT - Dispositions communes - Droit conventionnel - États et territoires non coopératifs - Constitution et mise à jour de la liste des États et territoires non coopératifs'},
  {'rank': 2,
   'score': 47.22,
   'boi_reference': 'BOI-INT-DG-20-50-20210224',
   'chunk_id': '5334-PGP__document_title',
   'chunk_kind': 'document_title',
   'section_path': 'INT - Dispositions communes - Droit conventionnel - États et territoires non coopératifs'},

In [8]:
doc_trace("q30")


{'query_id': 'q30',
 'query': 'prélèvement forfaitaire obligatoire applicable aux produits de placement à revenu fixe',
 'pattern': 'topic_specific',
 'expected_boi': 'BOI-RPPM-RCM-30-20-40-20220630',
 'returned_boi': ['BOI-RPPM-RCM-30-20-70-20191220',
  'BOI-RPPM-RCM-30-20-40-20220630',
  'BOI-RPPM-RCM-30-10-20-40-20140211',
  'BOI-RPPM-RCM-30-10-20-10-20191220',
  'BOI-RPPM-RCM-30-10-20-30-20140211'],
 'top_hits': [{'rank': 1,
   'score': 44.2023,
   'boi_reference': 'BOI-RPPM-RCM-30-20-70-20191220',
   'chunk_id': '7052-PGP__document_title',
   'chunk_kind': 'document_title',
   'section_path': "RPPM - Revenus de capitaux mobiliers, gains et profits assimilés - Modalités particulières d'imposition - Prélèvement forfaitaire obligatoire non libératoire de l'impôt sur le revenu applicable aux produits de placement à revenu fixe, aux produits et gains de cession de bons ou contrats de capitalisation et placements de même nature attachés à des primes versées à compter du 27 septembre 201

In [9]:
doc_trace("q01")


{'query_id': 'q01',
 'query': 'une JEI peut-elle demander le remboursement immédiat du CIR',
 'pattern': 'near_neighbor',
 'expected_boi': 'BOI-BIC-CHAMP-80-20-20-20-20240703',
 'returned_boi': ['BOI-RES-SJ-000014-20210309',
  'BOI-TVA-SECT-80-60-10-50-20120912',
  'BOI-ENR-DG-50-20-20160203',
  'BOI-TVA-DED-50-20-10-20150506',
  'BOI-TVA-DED-50-20-20-20210224'],
 'top_hits': [{'rank': 1,
   'score': 11.7225,
   'boi_reference': 'BOI-RES-SJ-000014-20210309',
   'chunk_id': '11456-PGP__document_title',
   'chunk_kind': 'document_title',
   'section_path': "RES - Sécurité juridique - Garantie contre les changements de position de l'administration fiscale - Délai de dépôt d'une demande de rescrit relatif aux jeunes entreprises innovantes (JEI) - (LPF, art. L 80 B, 4°)"},
  {'rank': 2,
   'score': 10.908,
   'boi_reference': 'BOI-TVA-SECT-80-60-10-50-20120912',
   'chunk_id': '882-PGP__document_title',
   'chunk_kind': 'document_title',
   'section_path': 'TVA - Régimes sectoriels - Agricu

## 5. Objectif numéro `#2`: le bon extrait du bon document

On compare:
- stage 1 document retrieval
- stage 2 top chunks
- stage 3 deep dive local borné


In [10]:
deep_dive_trace("q02")


{'query_id': 'q02',
 'query': "quelles dépenses de fonctionnement sont éligibles au crédit d'impôt recherche",
 'expected_boi': 'BOI-BIC-RICI-10-10-20-25-20250813',
 'expected_behavior': 'answer',
 'stage1_docs': [{'rank': 1,
   'score': 39.793,
   'boi_reference': 'BOI-BIC-RICI-10-10-20-25-20250813',
   'title': 'BIC - Réductions et crédits d’impôt - Crédits d’impôt - Crédit d’impôt recherche - Dépenses de recherche éligibles - Autres dépenses de fonctionnement'},
  {'rank': 2,
   'score': 31.7873,
   'boi_reference': 'BOI-BIC-RICI-10-10-20-50-20250813',
   'title': 'BIC - Réductions et crédits d’impôt - Crédits d’impôt - Crédit d’impôt recherche - Dépenses de recherche éligibles - Dépenses de normalisation afférentes aux produits de l’entreprise'},
  {'rank': 3,
   'score': 29.8877,
   'boi_reference': 'BOI-BIC-RICI-10-10-20-40-20250813',
   'title': 'BIC - Réductions et crédits d’impôt - Crédits d’impôt - Crédit d’impôt recherche - Dépenses de recherche éligibles relatives aux breve

In [11]:
deep_dive_trace("q04")


{'query_id': 'q04',
 'query': "extension du délai de reprise pour l'imposition des revenus de 2018 prélèvement à la source",
 'expected_boi': 'BOI-IR-PAS-50-20-50-20200210',
 'expected_behavior': 'answer',
 'stage1_docs': [{'rank': 1,
   'score': 56.1877,
   'boi_reference': 'BOI-IR-PAS-50-20-50-20200210',
   'title': "IR - Prélèvement à la source de l'impôt sur le revenu - Mesures transitoires - Autres mesures transitoires - Extension du délai de reprise pour l'imposition des revenus de 2018"},
  {'rank': 2,
   'score': 25.9773,
   'boi_reference': 'BOI-CF-PGR-10-70-20161229',
   'title': "CF - Prescription du droit de reprise de l'administration - Prorogation du délai de reprise en cas d'activités occultes ou de procès-verbal pour flagrance fiscale et conséquences sur certains délais"},
  {'rank': 3,
   'score': 25.9532,
   'boi_reference': 'BOI-IR-PAS-20-30-30-20180515',
   'title': "IR - Prélèvement à la source de l'impôt sur le revenu - Calcul du prélèvement - Actualisation du pré

In [12]:
deep_dive_trace("q11")


{'query_id': 'q11',
 'query': 'taux de TVA pour un système qui fixe un fauteuil roulant à une trottinette électrique',
 'expected_boi': 'BOI-RES-TVA-000074-20210309',
 'expected_behavior': 'answer',
 'stage1_docs': [{'rank': 1,
   'score': 60.1241,
   'boi_reference': 'BOI-RES-TVA-000074-20210309',
   'title': 'RES - Taxe sur la valeur ajoutée - Liquidation - Taux de TVA applicable aux systèmes de fixation permettant d’accrocher un fauteuil roulant à une trottinette électrique'},
  {'rank': 2,
   'score': 19.9509,
   'boi_reference': 'BOI-IR-RICI-390-20230629',
   'title': "IR - Réductions et crédits d'impôt - Crédit d’impôt pour le premier abonnement à un journal, à une publication périodique ou à un service de presse en ligne d'information politique et générale"},
  {'rank': 3,
   'score': 18.4469,
   'boi_reference': 'BOI-RES-TPS-000060-20210309',
   'title': "RES - Taxes et particpations sur les salaires - Taxe sur les salaires - Exonération applicable aux établissements d'enseigne

In [13]:
deep_dive_trace("q12")


{'query_id': 'q12',
 'query': 'détermination du redevable de la TVA pour les livraisons de biens et prestations de services',
 'expected_boi': 'BOI-TVA-DECLA-10-10-20-20251022',
 'expected_behavior': 'answer',
 'stage1_docs': [{'rank': 1,
   'score': 52.6082,
   'boi_reference': 'BOI-TVA-DECLA-10-10-20-20251022',
   'title': 'TVA - Régimes d’imposition et obligations déclaratives et comptables - Redevable de la taxe - Livraisons de biens et prestations de services - Détermination du redevable'},
  {'rank': 2,
   'score': 46.868,
   'boi_reference': 'BOI-TVA-DECLA-10-10-20200323',
   'title': "TVA - Régimes d'imposition et obligations déclaratives et comptables - Redevable de la taxe - Livraisons de biens et prestations de services"},
  {'rank': 3,
   'score': 43.279,
   'boi_reference': 'BOI-TVA-DECLA-10-10-30-20-20200902',
   'title': "TVA - Régimes d'imposition et obligations déclaratives et comptables - Redevable de la taxe - Livraisons de biens et prestations de services - Solidari

In [14]:
deep_dive_trace("q14")


{'query_id': 'q14',
 'query': "compensation et substitution de base légale en contentieux de l'assiette",
 'expected_boi': 'BOI-CTX-DG-20-40-10-20120912',
 'expected_behavior': 'answer',
 'stage1_docs': [{'rank': 1,
   'score': 47.3713,
   'boi_reference': 'BOI-CTX-DG-20-40-10-20120912',
   'title': "CTX - Contentieux de l'assiette de l'impôt - Dispositions communes - Compensation et substitution de base légale susceptibles d'être opposées par l'administration"},
  {'rank': 2,
   'score': 22.1031,
   'boi_reference': 'BOI-CTX-DG-10-10-20120912',
   'title': "CTX - Contentieux de l'assiette de l'impôt – Nature et place du contentieux de l'assiette de l'impôt"},
  {'rank': 3,
   'score': 19.514,
   'boi_reference': 'BOI-CTX-DG-20-60-20120912',
   'title': "CTX - Contentieux de l'assiette de l'impôt - Dispositions communes - Question prioritaire de constitutionnalité et questions préjudicielles"}],
 'stage2_top_chunks': [{'global_rank': 1,
   'document_rank': 1,
   'document_score': 47.37

In [15]:
deep_dive_trace("q29")


{'query_id': 'q29',
 'query': 'États et territoires non coopératifs droit conventionnel',
 'expected_boi': 'BOI-INT-DG-20-50-20210224',
 'expected_behavior': 'answer',
 'stage1_docs': [{'rank': 1,
   'score': 47.7398,
   'boi_reference': 'BOI-INT-DG-20-50-10-20210224',
   'title': 'INT - Dispositions communes - Droit conventionnel - États et territoires non coopératifs - Constitution et mise à jour de la liste des États et territoires non coopératifs'},
  {'rank': 2,
   'score': 47.22,
   'boi_reference': 'BOI-INT-DG-20-50-20210224',
   'title': 'INT - Dispositions communes - Droit conventionnel - États et territoires non coopératifs'},
  {'rank': 3,
   'score': 13.6365,
   'boi_reference': 'BOI-INT-DG-20-20-40-20120912',
   'title': "INT -Dispositions communes - Droit conventionnel - Modalités d'imposition au regard du droit conventionnel - Revenus immobiliers - Gains en capital - Professions indépendantes - Revenus d'emploi – Tantièmes"}],
 'stage2_top_chunks': [{'global_rank': 1,
  

In [16]:
deep_dive_trace("q30")


{'query_id': 'q30',
 'query': 'prélèvement forfaitaire obligatoire applicable aux produits de placement à revenu fixe',
 'expected_boi': 'BOI-RPPM-RCM-30-20-40-20220630',
 'expected_behavior': 'answer',
 'stage1_docs': [{'rank': 1,
   'score': 44.2023,
   'boi_reference': 'BOI-RPPM-RCM-30-20-70-20191220',
   'title': "RPPM - Revenus de capitaux mobiliers, gains et profits assimilés - Modalités particulières d'imposition - Prélèvement forfaitaire obligatoire non libératoire de l'impôt sur le revenu applicable aux produits de placement à revenu fixe, aux produits et gains de cession de bons ou contrats de capitalisation et placements de même nature attachés à des primes versées à compter du 27 septembre 2017, et aux revenus distribués - Mesures de contrôle applicables aux produits de placement à revenu fixe"},
  {'rank': 2,
   'score': 40.1469,
   'boi_reference': 'BOI-RPPM-RCM-30-20-40-20220630',
   'title': "RPPM - Revenus de capitaux mobiliers, gains et profits assimilés - Modalités p

## 6. Cas non supportés et fausses prémisses

Ici on regarde ce que le système fait aujourd'hui quand il devrait soit:
- s'abstenir
- ou retrouver un document pour corriger la prémisse


In [17]:
two_stage_trace("u01")


{'query_id': 'u01',
 'query': 'taux de TVA des cryptomonnaies minées par un particulier en 2026',
 'expected_boi': None,
 'expected_behavior': 'abstain',
 'stage1_docs': [{'rank': 1,
   'score': 21.7344,
   'boi_reference': 'BOI-TVA-LIQ-40-10-20241127',
   'title': 'TVA - Liquidation - Taux - Produits imposables au taux particulier de 2,10 %'},
  {'rank': 2,
   'score': 15.2049,
   'boi_reference': 'BOI-RES-TVA-000074-20210309',
   'title': 'RES - Taxe sur la valeur ajoutée - Liquidation - Taux de TVA applicable aux systèmes de fixation permettant d’accrocher un fauteuil roulant à une trottinette électrique'},
  {'rank': 3,
   'score': 14.5695,
   'boi_reference': 'BOI-TVA-GEO-10-10-20241120',
   'title': 'TVA - Régimes territoriaux - Régime de la Corse - Taxation particulière - Taux de la TVA applicables en Corse'}],
 'stage2_chunks': [{'global_rank': 1,
   'document_rank': 1,
   'document_score': 21.7344,
   'local_rank': 1,
   'local_score': 6.6231,
   'boi_reference': 'BOI-TVA-LIQ-

In [18]:
two_stage_trace("u03")


{'query_id': 'u03',
 'query': 'procédure fiscale applicable aux successions ouvertes au Luxembourg sans aucun lien avec la France',
 'expected_boi': None,
 'expected_behavior': 'abstain',
 'stage1_docs': [{'rank': 1,
   'score': 22.4527,
   'boi_reference': 'BOI-BIC-BASE-90-20180704',
   'title': "BIC - Base d'imposition - Exclusion du résultat net des produits et des charges sans lien avec l'activité professionnelle - Suppression des effets fiscaux de la théorie du bilan"},
  {'rank': 2,
   'score': 19.2683,
   'boi_reference': 'BOI-INT-CVB-LUX-20-20210223',
   'title': "INT - Convention fiscale entre la France et le Luxembourg - Règles d'imposition prévues pour certains revenus"},
  {'rank': 3,
   'score': 15.5943,
   'boi_reference': 'BOI-INT-CVB-CHE-20-20141224',
   'title': "INT - Convention fiscale entre la France et la Suisse en matière d'impôts sur les successions"}],
 'stage2_chunks': [{'global_rank': 1,
   'document_rank': 1,
   'document_score': 22.4527,
   'local_rank': 1,


In [19]:
two_stage_trace("u06")


{'query_id': 'u06',
 'query': "quelle convention fiscale entre la France et Mars s'applique aux salaires des astronautes",
 'expected_boi': None,
 'expected_behavior': 'abstain',
 'stage1_docs': [{'rank': 1,
   'score': 22.783,
   'boi_reference': 'BOI-INT-CVB-SGP-20170807',
   'title': 'INT - Convention fiscale entre la France et Singapour'},
  {'rank': 2,
   'score': 22.783,
   'boi_reference': 'BOI-INT-CVB-MYT-20160608',
   'title': 'INT - Convention fiscale entre la France et Mayotte'},
  {'rank': 3,
   'score': 22.783,
   'boi_reference': 'BOI-INT-CVB-ISR-20120912',
   'title': 'INT - Convention fiscale entre la France et Israël'}],
 'stage2_chunks': [{'global_rank': 1,
   'document_rank': 1,
   'document_score': 22.783,
   'local_rank': 1,
   'local_score': 2.4863,
   'boi_reference': 'BOI-INT-CVB-SGP-20170807',
   'chunk_id': '1725-PGP__section_window__1725-pgp-paragraph-00000__1725-pgp-paragraph-00008__int-convention-fiscale-entre-la-france-et-singapour',
   'chunk_kind': 'para

In [20]:
two_stage_trace("u17")


{'query_id': 'u17',
 'query': 'le BOI indique-t-il que le PEA autorise toujours les crypto-actifs',
 'expected_boi': None,
 'expected_behavior': 'answer',
 'stage1_docs': [{'rank': 1,
   'score': 15.2081,
   'boi_reference': 'BOI-PAT-IFI-30-10-30-30-20180608',
   'title': "PAT - IFI - Actifs exonérés - Exonération des actifs professionnels - Actifs affectés à la profession exercée dans le cadre d'une société soumise à l'impôt sur les sociétés - Opérations de fusion, scission de sociétés, apports partiels d'actifs et apports à titre pur et simple"},
  {'rank': 2,
   'score': 11.5009,
   'boi_reference': 'BOI-PAT-IFI-30-10-20180608',
   'title': 'PAT - IFI - Actifs exonérés - Exonération des actifs professionnels'},
  {'rank': 3,
   'score': 11.2603,
   'boi_reference': 'BOI-RES-IS-000028-20210309',
   'title': "RES - Impôt sur les sociétés - Fusions et opérations assimilées - Régime spécial des fusions - Cession ou apport de plein droit de titres grevés de l'engagement de conservation d

In [21]:
two_stage_trace("u23")


{'query_id': 'u23',
 'query': "le BOFIP indique-t-il qu'une JEI bénéficie automatiquement du remboursement immédiat de tout crédit d'impôt",
 'expected_boi': None,
 'expected_behavior': 'answer',
 'stage1_docs': [{'rank': 1,
   'score': 16.8288,
   'boi_reference': 'BOI-IR-RICI-390-20230629',
   'title': "IR - Réductions et crédits d'impôt - Crédit d’impôt pour le premier abonnement à un journal, à une publication périodique ou à un service de presse en ligne d'information politique et générale"},
  {'rank': 2,
   'score': 16.5696,
   'boi_reference': 'BOI-IR-PAS-50-10-40-20180704',
   'title': "IR - Prélèvement à la source de l'impôt sur le revenu - Mesures transitoires - Crédit d'impôt pour la modernisation du recouvrement - Crédit d'impôt applicable en matière de contributions et prélèvements sociaux"},
  {'rank': 3,
   'score': 16.3822,
   'boi_reference': 'BOI-RES-SJ-000014-20210309',
   'title': "RES - Sécurité juridique - Garantie contre les changements de position de l'administ

## 7. Audit d'abstention

Important:
- la gate actuelle n'est pas encore une solution finale
- c'est un audit transparent des signaux disponibles
- le meilleur rule actuel repose sur le taux de tokens de requête non couverts par `titre du doc + top chunk`


In [22]:
{
    "best_rule": abstention_v2["best_rule"],
    "top_candidate_rules": abstention_v2["top_candidate_rules"][:8],
}


{'best_rule': {'rule_family': 'combined_uncovered_ratio',
  'uncovered_ratio_min': 0.5,
  'accuracy': 0.8267,
  'abstain_precision': 0.7778,
  'abstain_recall': 0.3889,
  'answer_recall': 0.9649,
  'tp': 7,
  'tn': 55,
  'fp': 2,
  'fn': 11},
 'top_candidate_rules': [{'rule_family': 'combined_uncovered_ratio',
   'uncovered_ratio_min': 0.5,
   'accuracy': 0.8267,
   'abstain_precision': 0.7778,
   'abstain_recall': 0.3889,
   'answer_recall': 0.9649,
   'tp': 7,
   'tn': 55,
   'fp': 2,
   'fn': 11},
  {'rule_family': 'combined_uncovered_ratio',
   'uncovered_ratio_min': 0.3,
   'accuracy': 0.8,
   'abstain_precision': 0.5517,
   'abstain_recall': 0.8889,
   'answer_recall': 0.7719,
   'tp': 16,
   'tn': 44,
   'fp': 13,
   'fn': 2},
  {'rule_family': 'combined_uncovered_ratio',
   'uncovered_ratio_min': 0.4,
   'accuracy': 0.7733,
   'abstain_precision': 0.5263,
   'abstain_recall': 0.5556,
   'answer_recall': 0.8421,
   'tp': 10,
   'tn': 48,
   'fp': 9,
   'fn': 8},
  {'rule_family'

In [23]:
{
    "correct_abstentions": [
        {
            "id": row["id"],
            "combined_uncovered_ratio": row["combined_uncovered_ratio"],
            "query": row["query"],
        }
        for row in abstention_v2["rows"]
        if row["expected_behavior"] == "abstain" and row["predicted_behavior"] == "abstain"
    ][:10],
    "wrong_answers_should_abstain": [
        {
            "id": row["id"],
            "combined_uncovered_ratio": row["combined_uncovered_ratio"],
            "query": row["query"],
        }
        for row in abstention_v2["rows"]
        if row["expected_behavior"] == "abstain" and row["predicted_behavior"] == "answer"
    ][:10],
    "wrong_abstentions_should_answer": [
        {
            "id": row["id"],
            "combined_uncovered_ratio": row["combined_uncovered_ratio"],
            "query": row["query"],
        }
        for row in abstention_v2["rows"]
        if row["expected_behavior"] == "answer" and row["predicted_behavior"] == "abstain"
    ][:10],
}


{'correct_abstentions': [{'id': 'u07',
   'combined_uncovered_ratio': 0.5,
   'query': 'taxe applicable aux fusées de tourisme spatial au départ de Paris'},
  {'id': 'u09',
   'combined_uncovered_ratio': 0.5,
   'query': 'TVA 2026 sur la vente de NFT par un mineur non émancipé'},
  {'id': 'u10',
   'combined_uncovered_ratio': 0.5385,
   'query': "procédure de contestation d'un impôt fédéral canadien devant l'administration fiscale française"},
  {'id': 'u11',
   'combined_uncovered_ratio': 0.5,
   'query': 'TVA applicable en 2030 aux cryptomonnaies minées par une intelligence artificielle autonome'},
  {'id': 'u12',
   'combined_uncovered_ratio': 0.6,
   'query': 'barème 2027 de la taxe sur les capsules spatiales privées'},
  {'id': 'u13',
   'combined_uncovered_ratio': 0.5385,
   'query': "procédure de contestation d'un impôt communal belge devant l'administration fiscale française"},
  {'id': 'u16',
   'combined_uncovered_ratio': 0.5455,
   'query': 'taxe française sur la revente de 

In [24]:
abstention_trace("u01")


{'query_id': 'u01',
 'query': 'taux de TVA des cryptomonnaies minées par un particulier en 2026',
 'pattern': 'unsupported',
 'expected_behavior': 'abstain',
 'predicted_behavior': 'answer',
 'decision_correct': False,
 'top_doc_boi': 'BOI-TVA-LIQ-40-10-20241127',
 'doc_margin': 6.5295,
 'title_overlap_ratio': 0.3636,
 'chunk_overlap_ratio': 0.7273,
 'combined_uncovered_ratio': 0.2727,
 'top_chunk_text': 'Ne peuvent être considérés comme des matières premières utilisées pour la fabrication de médicaments : la gélatine utilisée pour la fabrication de gélules renfermant des médicaments qui sont des formes pharmaceutiques. Cette gélatine présente en effet les mêmes caractéristiques essentielles que celle qui est utilisée p'}

In [25]:
abstention_trace("u03")


{'query_id': 'u03',
 'query': 'procédure fiscale applicable aux successions ouvertes au Luxembourg sans aucun lien avec la France',
 'pattern': 'unsupported',
 'expected_behavior': 'abstain',
 'predicted_behavior': 'answer',
 'decision_correct': False,
 'top_doc_boi': 'BOI-BIC-BASE-90-20180704',
 'doc_margin': 3.1844,
 'title_overlap_ratio': 0.2857,
 'chunk_overlap_ratio': 0.5714,
 'combined_uncovered_ratio': 0.4286,
 'top_chunk_text': "S'agissant de l'appréciation du caractère professionnel de l'exercice de la location meublée, il convient de se reporter au II § 40 et suivants du BOI-BIC-CHAMP-40-10 .\n\nSur le plan comptable, l’entrepreneur individuel, titulaire de bénéfices industriels et commerciaux ou de bénéfices agricoles dispose d’une liberté d’"}

In [26]:
abstention_trace("u17")


{'query_id': 'u17',
 'query': 'le BOI indique-t-il que le PEA autorise toujours les crypto-actifs',
 'pattern': 'false_premise',
 'expected_behavior': 'answer',
 'predicted_behavior': 'abstain',
 'decision_correct': False,
 'top_doc_boi': 'BOI-PAT-IFI-30-10-30-30-20180608',
 'doc_margin': 3.7072,
 'title_overlap_ratio': 0.25,
 'chunk_overlap_ratio': 0.3333,
 'combined_uncovered_ratio': 0.6667,
 'top_chunk_text': "Dans une telle hypothèse, les règles relatives aux actifs professionnels trouvent à s'appliquer, y compris pour l'exercice de la création de l'entreprise, dans les conditions de droit commun.\n\nIl convient dans cette hypothèse de transposer les règles mentionnées ci-dessus II-A § 110 et suivants applicables au cas où la"}

In [27]:
abstention_trace("u23")


{'query_id': 'u23',
 'query': "le BOFIP indique-t-il qu'une JEI bénéficie automatiquement du remboursement immédiat de tout crédit d'impôt",
 'pattern': 'false_premise',
 'expected_behavior': 'answer',
 'predicted_behavior': 'answer',
 'decision_correct': True,
 'top_doc_boi': 'BOI-IR-RICI-390-20230629',
 'doc_margin': 0.2593,
 'title_overlap_ratio': 0.3333,
 'chunk_overlap_ratio': 0.5556,
 'combined_uncovered_ratio': 0.4444,
 'top_chunk_text': 'Lorsqu’il est mis fin à l’abonnement avant la durée minimale de douze mois prévue au I-B-4-b § 100 à 120 , le crédit d’impôt fait l’objet d’une reprise au titre de l’année de la résiliation, en application du IV de l’ article 200 sexdecies du CGI . Il est intégralement remis en cause, sans application d’un quelconque p'}

## 8. Conclusion d'ingénieur

Lecture stricte:
- `#1` bon document: oui, c'est maintenant la partie la plus solide
- `#2` bon extrait: meilleur qu'avant grâce au stage 2 `body` et au stage 3 borné, mais encore à auditer
- abstention: encore insuffisante pour servir de gate finale seule

Donc la suite rationnelle reste:
- renforcer encore l'audit du retrieval documentaire sur les voisinages restants
- continuer à améliorer la qualité du passage local
- ne brancher le LLM qu'après une gate d'abstention plus crédible
